# JupyterLite weight sensitivity demo

Monte Carlo uncertainty for the combined risk score using synthetic component values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n = 20
features = pd.DataFrame({
    "facility_id": [f"HF{i:03d}" for i in range(n)],
    "spillover_suitability": rng.beta(2, 3, n),
    "human_exposure": rng.beta(2.5, 2.5, n),
    "importation_pressure": rng.beta(1.5, 4, n),
    "facility_vulnerability": rng.beta(2, 2, n),
})
base = np.array([0.25, 0.25, 0.30, 0.20])
cols = ["spillover_suitability", "human_exposure", "importation_pressure", "facility_vulnerability"]
iterations = 500
scores = []
for _ in range(iterations):
    w = rng.dirichlet(base * 40)
    scores.append((features[cols].to_numpy() * w).sum(axis=1))
arr = np.vstack(scores)
features["risk_mean"] = arr.mean(axis=0)
features["risk_p05"] = np.quantile(arr, 0.05, axis=0)
features["risk_p95"] = np.quantile(arr, 0.95, axis=0)
features.sort_values("risk_mean", ascending=False).head(10)


In [ ]:
ranked = features.sort_values("risk_mean", ascending=True)
plt.figure(figsize=(8, 6))
plt.errorbar(
    ranked["risk_mean"], ranked["facility_id"],
    xerr=[ranked["risk_mean"] - ranked["risk_p05"], ranked["risk_p95"] - ranked["risk_mean"]],
    fmt="o"
)
plt.xlabel("Risk score with 90% uncertainty interval")
plt.title("Synthetic weight-sensitivity intervals")
plt.tight_layout()
plt.show()
